# Model Training & Single Image Inference Pipeline
## Klasifikasi Potensi Disgrafia Berdasarkan Citra Tulisan Tangan

Notebook ini mencakup dua alur kerja utama:
1. **Model Training & Evaluasi**: Melatih beberapa algoritma Machine Learning menggunakan dataset fitur (`dysgraphia_features.csv`) dengan evaluasi **Stratified 5-Fold Cross Validation**, memilih model terbaik, dan menyimpannya (`.pkl`).
2. **Inference Pipeline (Prediksi Citra Baru)**: Fungsi mandiri yang menerima **input 1 file gambar baru**, melakukan pra-pemrosesan citra, mengekstrak 25 fitur grafologi secara otomatis, dan memberikan output prediksi (`Potential Dysgraphia` vs `Low Potential Dysgraphia`) lengkap dengan nilai probabilitas keyakinan (*confidence score*) serta kartu diagnosis visual.

---
### 📌 Alur Kerja Sistem (End-to-End Pipeline)

```
[Input: Gambar Tulisan Tangan Baru (.jpg/.png)]
                   │
                   ▼
       1. Preprocessing Citra (Otsu Binarization & Crop)
                   │
                   ▼
       2. Feature Extraction (Ekstraksi 25 Fitur Disgrafia)
                   │
                   ▼
       3. Feature Scaling (Menggunakan Standard Scaler)
                   │
                   ▼
       4. Model Prediction (Best ML Model)
                   │
                   ▼
[Output: Prediksi Label + Tingkat Keyakinan (%) + Kartu Diagnosis Visual]
```


---
## 1. Import Libraries
Memuat library Machine Learning, pemrosesan citra, dan penyimpanan model.


In [ ]:
import os
import math
import random
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Library Feature Extraction
from scipy import stats
from scipy.ndimage import distance_transform_edt
from skimage.feature import graycomatrix, graycoprops

# Library Machine Learning & Evaluation
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# Algoritma Klasifikasi
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier

try:
    from IPython.display import display
except ImportError:
    display = print

# Konfigurasi visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100

# Random Seed
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("[INFO] Seluruh pustaka Machine Learning & Inference siap digunakan.")


---
## 2. Memuat Dataset Fitur (`dysgraphia_features.csv`)
Memuat dataset tabular berisi 25 fitur grafologi dan computer vision yang telah diekstrak sebelumnya.


In [ ]:
csv_file = "dysgraphia_features.csv"

if not os.path.exists(csv_file):
    raise FileNotFoundError(f"File '{csv_file}' tidak ditemukan. Pastikan Anda telah menjalankan ekstraksi fitur terlebih dahulu.")

df = pd.read_csv(csv_file)
print(f"[INFO] Dataset fitur berhasil dimuat: {df.shape[0]} baris, {df.shape[1]} kolom.")

feature_cols = [c for c in df.columns if c not in ['filename', 'label', 'target']]
print(f"[INFO] Jumlah kolom fitur: {len(feature_cols)}")
print(f"[INFO] Distribusi target:")
print(df['label'].value_counts())

X = df[feature_cols].values
y = df['target'].values  # 0: Low Potential Dysgraphia, 1: Potential Dysgraphia
label_names = {0: 'Low Potential Dysgraphia', 1: 'Potential Dysgraphia'}


---
## 3. Pelatihan Model & Evaluasi Perbandingan (Stratified 5-Fold CV)
Kita membandingkan 6 algoritma klasifikasi Machine Learning menggunakan **Stratified 5-Fold Cross Validation** agar hasil evaluasi objektif dan bebas dari bias *data leakage*:
1. **Logistic Regression** (L2 Regularized)
2. **Support Vector Classifier (SVC - RBF Kernel)**
3. **Random Forest Classifier**
4. **Extra Trees Classifier**
5. **Gradient Boosting Classifier**
6. **K-Nearest Neighbors (KNN)**


In [ ]:
# Definisi Model-model yang akan diuji
models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    'Support Vector Machine (RBF)': SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=6, random_state=RANDOM_STATE),
    'Extra Trees': ExtraTreesClassifier(n_estimators=100, max_depth=6, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=RANDOM_STATE),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

# Standarisasi Fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Stratified 5-Fold Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = []
cv_predictions = {}
cv_probabilities = {}

print("[INFO] Memulai evaluasi 5-Fold Cross Validation untuk seluruh model...")

for name, model in models.items():
    # Metrik evaluasi silang
    scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
    cv_res = cross_validate(model, X_scaled, y, cv=cv, scoring=scoring)
    
    # Simpan prediksi Out-of-Fold untuk Confusion Matrix
    y_pred = cross_val_predict(model, X_scaled, y, cv=cv)
    y_prob = cross_val_predict(model, X_scaled, y, cv=cv, method='predict_proba')[:, 1]
    
    cv_predictions[name] = y_pred
    cv_probabilities[name] = y_prob
    
    results.append({
        'Model': name,
        'Accuracy (%)': round(np.mean(cv_res['test_accuracy']) * 100, 2),
        'Precision (%)': round(np.mean(cv_res['test_precision']) * 100, 2),
        'Recall (%)': round(np.mean(cv_res['test_recall']) * 100, 2),
        'F1-Score (%)': round(np.mean(cv_res['test_f1']) * 100, 2),
        'ROC-AUC (%)': round(np.mean(cv_res['test_roc_auc']) * 100, 2),
        'Std Acc (%)': round(np.std(cv_res['test_accuracy']) * 100, 2)
    })

comparison_df = pd.DataFrame(results).sort_values(by='F1-Score (%)', ascending=False).reset_index(drop=True)
print("\n================ HASIL PERBANDINGAN MODEL (5-FOLD CV) ================")
display(comparison_df)


---
## 4. Visualisasi Kinerja Model Terbaik
Menampilkan **Confusion Matrix** dan **Kurva ROC (Receiver Operating Characteristic)** untuk model dengan performa tertinggi.


In [ ]:
best_model_name = comparison_df.iloc[0]['Model']
model_name = best_model_name  # Alias agar kompatibel
print(f"[INFO] Model Terbaik Terpilih: '{best_model_name}'")

best_y_pred = cv_predictions[best_model_name]
best_y_prob = cv_probabilities[best_model_name]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Confusion Matrix
cm = confusion_matrix(y, best_y_pred)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
    xticklabels=['Low Potential (0)', 'Potential (1)'],
    yticklabels=['Low Potential (0)', 'Potential (1)'],
    cbar=False, annot_kws={'size': 14, 'weight': 'bold'}
)
axes[0].set_title(f'Confusion Matrix: {best_model_name}', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Prediksi Model', fontweight='bold')
axes[0].set_ylabel('Label Aktual', fontweight='bold')

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y, best_y_prob)
auc_val = roc_auc_score(y, best_y_prob)

axes[1].plot(fpr, tpr, color='#d95f02', lw=2.5, label=f'{best_model_name} (AUC = {auc_val:.3f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1.5, label='Random Guess (AUC = 0.500)')
axes[1].set_title('Receiver Operating Characteristic (ROC) Curve', fontsize=12, fontweight='bold')
axes[1].set_xlabel('False Positive Rate (1 - Specificity)', fontweight='bold')
axes[1].set_ylabel('True Positive Rate (Sensitivity / Recall)', fontweight='bold')
axes[1].legend(loc='lower right', frameon=True)
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# Laporan Klasifikasi Rinci
print("\nClassification Report Rinci:")
print(classification_report(y, best_y_pred, target_names=['Low Potential Dysgraphia', 'Potential Dysgraphia']))


---
## 5. Melatih Final Model & Menyimpan File Model (`.pkl`)
Setelah model terbaik terpilih melalui Cross Validation, kita melatihnya pada seluruh data fitur dan menyimpannya bersama dengan `StandardScaler` dan daftar nama fitur.


In [ ]:
best_model_instance = models[best_model_name]
# Fit pada seluruh data
best_model_instance.fit(X_scaled, y)

# Simpan artefak model
MODEL_FILE = "best_dysgraphia_model.pkl"
SCALER_FILE = "dysgraphia_scaler.pkl"
FEATURES_FILE = "feature_names.json"

joblib.dump(best_model_instance, MODEL_FILE)
joblib.dump(scaler, SCALER_FILE)

import json
with open(FEATURES_FILE, 'w', encoding='utf-8') as f:
    json.dump(feature_cols, f, indent=2)

print(f"[SUCCESS] Model '{best_model_name}' berhasil disimpan ke '{MODEL_FILE}'")
print(f"[SUCCESS] Scaler berhasil disimpan ke '{SCALER_FILE}'")
print(f"[SUCCESS] Metadata fitur disimpan ke '{FEATURES_FILE}'")


---
## 6. Pipeline Inferensi Citra Tunggal (*Single Image Inference*)
Berikut adalah fungsi lengkap dan mandiri untuk menerima **input 1 gambar tulisan tangan baru**, mengekstrak fiturnya, dan memprediksi potensi disgrafia.


In [ ]:
def extract_features_from_image(img_path):
    """
    Fungsi preprocessing dan ekstraksi 25 fitur grafologi dari 1 file gambar baru.
    """
    # 1. Buka citra sebagai Grayscale
    img = Image.open(img_path).convert('L')
    gray = np.array(img)
    
    # 2. Otsu Binarization (Inverted: tinta = 255, kertas = 0)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # 3. Noise removal kecil
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    binary_clean = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    
    # 4. Tight crop
    coords = cv2.findNonZero(binary_clean)
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        pad = 4
        x1, y1 = max(0, x - pad), max(0, y - pad)
        x2, y2 = min(gray.shape[1], x + w + pad), min(gray.shape[0], y + h + pad)
        cropped_gray = gray[y1:y2, x1:x2]
        cropped_binary = binary_clean[y1:y2, x1:x2]
    else:
        cropped_gray = gray
        cropped_binary = binary_clean
        
    h_img, w_img = cropped_binary.shape
    
    # Connected Components
    num_labels, labels, stats_cc, centroids = cv2.connectedComponentsWithStats(cropped_binary, connectivity=8)
    min_area = 15
    valid_indices = [i for i in range(1, num_labels) if stats_cc[i, cv2.CC_STAT_AREA] >= min_area]
    if len(valid_indices) < 2:
        valid_indices = list(range(1, max(2, num_labels)))
        
    comp_widths = [stats_cc[i, cv2.CC_STAT_WIDTH] for i in valid_indices]
    comp_heights = [stats_cc[i, cv2.CC_STAT_HEIGHT] for i in valid_indices]
    comp_centroids = [centroids[i] for i in valid_indices]
    
    sorted_by_x = sorted(zip(comp_centroids, comp_widths, comp_heights), key=lambda item: item[0][0])
    sorted_centroids = [item[0] for item in sorted_by_x]
    
    # A. Baseline
    y_coords = [c[1] for c in sorted_centroids]
    baseline_dev_std = float(np.std(y_coords)) if len(y_coords) > 1 else 0.0
    y_diffs = np.abs(np.diff(y_coords)) if len(y_coords) > 1 else [0.0]
    baseline_waviness = float(np.mean(y_diffs))
    
    # B. Dimensions
    h_mean = float(np.mean(comp_heights)) if comp_heights else 0.0
    h_std = float(np.std(comp_heights)) if len(comp_heights) > 1 else 0.0
    height_cov = (h_std / h_mean) if h_mean > 0 else 0.0
    w_mean = float(np.mean(comp_widths)) if comp_widths else 0.0
    w_std = float(np.std(comp_widths)) if len(comp_widths) > 1 else 0.0
    width_cov = (w_std / w_mean) if w_mean > 0 else 0.0
    char_aspect_ratios = [w / h if h > 0 else 0 for w, h in zip(comp_widths, comp_heights)]
    aspect_ratio_char_mean = float(np.mean(char_aspect_ratios)) if char_aspect_ratios else 0.0
    aspect_ratio_char_std = float(np.std(char_aspect_ratios)) if len(char_aspect_ratios) > 1 else 0.0
    
    # C. Slant
    sobelx = cv2.Sobel(cropped_gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(cropped_gray, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = np.sqrt(sobelx**2 + sobely**2)
    edge_mask = (magnitude > np.percentile(magnitude, 75)) & (cropped_binary > 0)
    if np.sum(edge_mask) > 10:
        angles_rad = np.arctan2(sobely[edge_mask], sobelx[edge_mask])
        angles_deg = np.rad2deg(angles_rad)
        angles_deg = ((angles_deg + 90) % 180) - 90
        slant_angle_mean = float(np.mean(np.abs(angles_deg)))
        slant_angle_std = float(np.std(angles_deg))
    else:
        slant_angle_mean = 0.0
        slant_angle_std = 0.0
        
    # D. Spacing & Density
    total_pixels = h_img * w_img
    ink_pixels = int(np.sum(cropped_binary == 255))
    ink_density = float(ink_pixels / total_pixels) if total_pixels > 0 else 0.0
    spacings = []
    for k in range(len(sorted_by_x) - 1):
        x_curr = sorted_by_x[k][0][0] + (sorted_by_x[k][1] / 2.0)
        x_next = sorted_by_x[k+1][0][0] - (sorted_by_x[k+1][1] / 2.0)
        space = max(0, x_next - x_curr)
        spacings.append(space)
    inter_comp_spacing_mean = float(np.mean(spacings)) if spacings else 0.0
    inter_comp_spacing_std = float(np.std(spacings)) if len(spacings) > 1 else 0.0
    num_components_density = float(len(valid_indices) / (w_img / 1000.0)) if w_img > 0 else 0.0
    
    # E. Roughness & Thickness
    contours, _ = cv2.findContours(cropped_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    roughness_vals = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area >= min_area:
            perimeter = cv2.arcLength(cnt, True)
            r = (perimeter ** 2) / (4 * math.pi * area) if area > 0 else 0.0
            roughness_vals.append(r)
    stroke_roughness_mean = float(np.mean(roughness_vals)) if roughness_vals else 1.0
    stroke_roughness_std = float(np.std(roughness_vals)) if len(roughness_vals) > 1 else 0.0
    
    dist_map = distance_transform_edt(cropped_binary == 255)
    ink_distances = dist_map[cropped_binary == 255]
    if len(ink_distances) > 0:
        stroke_thickness_mean = float(np.mean(ink_distances) * 2.0)
        stroke_thickness_std = float(np.std(ink_distances) * 2.0)
    else:
        stroke_thickness_mean = 0.0
        stroke_thickness_std = 0.0
        
    # F. GLCM
    gray_resized = cv2.resize(cropped_gray, (256, 64), interpolation=cv2.INTER_AREA)
    gray_quantized = (gray_resized // 16).astype(np.uint8)
    glcm = graycomatrix(gray_quantized, distances=[1, 3], angles=[0, np.pi/4, np.pi/2], levels=16, symmetric=True, normed=True)
    glcm_contrast = float(np.mean(graycoprops(glcm, 'contrast')))
    glcm_dissimilarity = float(np.mean(graycoprops(glcm, 'dissimilarity')))
    glcm_homogeneity = float(np.mean(graycoprops(glcm, 'homogeneity')))
    glcm_energy = float(np.mean(graycoprops(glcm, 'energy')))
    glcm_correlation = float(np.mean(graycoprops(glcm, 'correlation')))
    
    features = {
        'baseline_dev_std': baseline_dev_std,
        'baseline_waviness': baseline_waviness,
        'height_mean': h_mean,
        'height_std': h_std,
        'height_cov': height_cov,
        'width_mean': w_mean,
        'width_std': w_std,
        'width_cov': width_cov,
        'aspect_ratio_char_mean': aspect_ratio_char_mean,
        'aspect_ratio_char_std': aspect_ratio_char_std,
        'slant_angle_mean': slant_angle_mean,
        'slant_angle_std': slant_angle_std,
        'ink_density': ink_density,
        'inter_comp_spacing_mean': inter_comp_spacing_mean,
        'inter_comp_spacing_std': inter_comp_spacing_std,
        'num_components_density': num_components_density,
        'stroke_roughness_mean': stroke_roughness_mean,
        'stroke_roughness_std': stroke_roughness_std,
        'stroke_thickness_mean': stroke_thickness_mean,
        'stroke_thickness_std': stroke_thickness_std,
        'glcm_contrast': glcm_contrast,
        'glcm_dissimilarity': glcm_dissimilarity,
        'glcm_homogeneity': glcm_homogeneity,
        'glcm_energy': glcm_energy,
        'glcm_correlation': glcm_correlation
    }
    return features, cropped_binary

print("[INFO] Fungsi ekstraksi fitur inferensi siap.")


---
## 7. Fungsi Prediksi Gambar Interaktif (*Diagnosis Function*)
Fungsi `predict_dysgraphia(image_path)` memuat model yang tersimpan, memproses gambar yang diberikan, dan menampilkan kartu hasil diagnosis lengkap dengan visualisasi citra, label hasil prediksi, probabilitas keyakinan (*confidence score*), serta metrik fitur utama.


In [ ]:
def predict_dysgraphia(image_path, model_path="best_dysgraphia_model.pkl", scaler_path="dysgraphia_scaler.pkl", show_card=True):
    """
    Memprediksi potensi disgrafia dari 1 berkas citra tulisan tangan.
    
    Parameters:
        image_path (str): Path ke file gambar (.jpg, .png, dll.)
        model_path (str): Path ke file model (.pkl)
        scaler_path (str): Path ke file scaler (.pkl)
        show_card (bool): Menampilkan visualisasi kartu diagnosis
        
    Returns:
        dict: Hasil prediksi berisi label, probabilitas, dan ringkasan fitur
    """
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"File gambar tidak ditemukan: {image_path}")
        
    # 1. Load model & scaler
    model = joblib.load(model_path)
    scaler_loaded = joblib.load(scaler_path)
    
    # 2. Ekstrak fitur & citra biner
    features_dict, binary_crop = extract_features_from_image(image_path)
    
    # 3. Urutkan fitur sesuai urutan saat training
    feature_vector = np.array([features_dict[col] for col in feature_cols]).reshape(1, -1)
    
    # 4. Standarisasi
    feature_scaled = scaler_loaded.transform(feature_vector)
    
    # 5. Prediksi
    pred_class = model.predict(feature_scaled)[0]
    prob_scores = model.predict_proba(feature_scaled)[0]
    
    pred_label = "Potential Dysgraphia" if pred_class == 1 else "Low Potential Dysgraphia"
    conf_score = prob_scores[pred_class] * 100
    prob_pot = prob_scores[1] * 100
    prob_low = prob_scores[0] * 100
    
    result = {
        'image_path': str(image_path),
        'predicted_class': int(pred_class),
        'model_name': 'Random Forest Classifier',
        'predicted_label': pred_label,
        'confidence_score': round(conf_score, 2),
        'prob_low_potential': round(prob_low, 2),
        'prob_potential': round(prob_pot, 2),
        'key_features': {
            'Baseline Waviness': round(features_dict['baseline_waviness'], 2),
            'Letter Height CoV': round(features_dict['height_cov'], 2),
            'Slant Variation (Std)': round(features_dict['slant_angle_std'], 2),
            'Stroke Roughness': round(features_dict['stroke_roughness_mean'], 2),
            'Ink Density': round(features_dict['ink_density'], 3)
        }
    }
    
    # Visualisasi Kartu Diagnosis
    if show_card:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={'width_ratios': [1.2, 1]})
        
        # Panel Kiri: Citra Input
        orig_img = Image.open(image_path)
        axes[0].imshow(orig_img, cmap='gray')
        axes[0].set_title(f"Input Citra Tulisan Tangan\n({Path(image_path).name})", fontsize=11, fontweight='bold')
        axes[0].axis('off')
        
        # Panel Kanan: Kartu Hasil Diagnosis
        axes[1].axis('off')
        badge_color = '#d95f02' if pred_class == 1 else '#2b5c8f'
        badge_text = "POTENSI DISGRAFIA" if pred_class == 1 else "POTENSI RENDAH DISGRAFIA"
        
        card_text = (
            f"=== HASIL DIAGNOSIS MODEL ===\n\n"
            f"Status Prediksi : {pred_label}\n"
            f"Tingkat Keyakinan : {conf_score:.1f}%\n\n"
            f"Rincian Probabilitas:\n"
            f" - Low Potential  : {prob_low:.1f}%\n"
            f" - Potential      : {prob_pot:.1f}%\n\n"
            f"Indikator Grafologi Utama:\n"
            f" • Baseline Waviness (Gelombang Baris) : {result['key_features']['Baseline Waviness']}\n"
            f" • Height Variation (Ketidakseragaman)  : {result['key_features']['Letter Height CoV']}\n"
            f" • Slant Variation (Ketidakkonsistenan): {result['key_features']['Slant Variation (Std)']}°\n"
            f" • Stroke Roughness (Kerutan Goresan)  : {result['key_features']['Stroke Roughness']}\n"
        )
        
        axes[1].text(
            0.05, 0.95, card_text, transform=axes[1].transAxes,
            fontsize=10.5, verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=0.8', facecolor='#f8f9fa', edgecolor=badge_color, linewidth=2.5)
        )
        
        plt.suptitle(f"Hasil Evaluasi Citra: [{badge_text}]", fontsize=13, fontweight='bold', color=badge_color, y=1.02)
        plt.tight_layout()
        plt.show()
        
    return result

print("[INFO] Fungsi predict_dysgraphia() berhasil didefinisikan dan siap dipakai.")


---
## 8. Uji Coba Prediksi pada Sampel Gambar Baru
Mari kita uji fungsi `predict_dysgraphia()` dengan memasukkan sampel gambar dari folder dataset untuk melihat bagaimana model memproses dan membedakan citra tulisan tangan.


In [1]:
# Contoh Uji Coba 1: Gambar dari kelas 'Low Potential Dysgraphia'
sample_low = "Low Potential Dysgraphia/LPD (1).jpg"
print(f"\n--- Menguji Gambar: {sample_low} ---")
res_1 = predict_dysgraphia(sample_low)

# Contoh Uji Coba 2: Gambar dari kelas 'Potential Dysgraphia'
sample_pot = "Potential Dysgraphia/PD (1).jpg"
print(f"\n--- Menguji Gambar: {sample_pot} ---")
res_2 = predict_dysgraphia(sample_pot)



--- Menguji Gambar: Low Potential Dysgraphia/LPD (1).jpg ---


NameError: name 'predict_dysgraphia' is not defined

---
## 9. Cara Menggunakan untuk Gambar Milik Anda Sendiri
Jika Anda memiliki file gambar tulisan tangan baru (misal hasil foto tulisan anak), Anda cukup memanggil fungsi:

```python
# Masukkan path file gambar Anda di sini:
my_image_path = "path/to/tulisan_baru.jpg"

# Jalankan prediksi
hasil = predict_dysgraphia(my_image_path)

# Hasil berisi dictionary:
print(hasil['predicted_label'])
print(hasil['confidence_score'])
```


In [2]:
# Template untuk memprediksi file gambar kustom Anda:
# Ganti nama file di bawah ini sesuai gambar yang ingin Anda uji:
test_custom_path = "Potential Dysgraphia/PD (10).jpg"

if os.path.exists(test_custom_path):
    hasil_prediksi = predict_dysgraphia(test_custom_path)
else:
    print(f"File {test_custom_path} tidak ditemukan.")


NameError: name 'os' is not defined